In [9]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [10]:
from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [11]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [12]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019dfec0-489d-7a30-a6ec-acb0afdf6fa7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '26c15aa2-c5c0-4e73-87ba-aa746dacbb94', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}}
Tool: get_weather
Tool: {'location': 'Boston'}


### Tool Execution loops

In [17]:
# Step 1: Model generates tool calls
messages = [
    {
        "role": "user",
        "content": "What's weather in New Delhi "
    }
]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass resutls back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in New Delhi is sunny.


In [16]:
messages

[{'role': 'user', 'content': "What's weather in New Delhi "},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New Delhi"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dfec1-622b-7123-a7e2-db8934742966-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New Delhi'}, 'id': 'd37995dd-a75b-4995-9f8b-dbcc0f0e00ba', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 16, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content="It's sunny in New Delhi", name='get_weather', tool_call_id='d37995dd-a75b-4995-9f8b-dbcc0f0e00ba')]